In [1]:
include("../RayTracing.jl")

Main.RayTracing

In [2]:
parsed_args = RayTracing.parse_commandline()
    
parsed_args["scene-number"] = 1

1

In [3]:
# set up logging
logger = RayTracing.setup_logging(parsed_args["debug"])
RayTracing.global_logger(logger)

# set random seed
RayTracing.Random.seed!(parsed_args["seed"])

Random.TaskLocalRNG()

In [4]:
I, scene = RayTracing.build_scene(parsed_args)
lights = scene.lights
wbounds = RayTracing.world_bounds(scene.b)


There are 91 objects in the scene, building BVH
  0.051457 seconds (89.89 k allocations: 6.293 MiB, 98.18% compilation time)
Done building BVH
Using 5 samples per pixel
There are 42 lights in the scene


Main.RayTracing.Bounds3([-978.8225099390856, -0.0001, -992.9646455628165], [300.0, 265.0, 300.0])

In [5]:
N = 10
N_voxels = N^3

1000

In [8]:
function calc_voxel_size(wbounds, N_voxels, max_recursion=100)
    size = floor((prod(wbounds.pMax - wbounds.pMin) / N_voxels)^(1/3))
    N = prod(floor.((wbounds.pMax - wbounds.pMin)/size) .+1)
    depth = 1
    while (N > N_voxels) & (depth <= max_recursion)
        depth += 1
        N = prod(floor.((wbounds.pMax - wbounds.pMin)/(size+depth)) .+1)
    end
    @assert prod(floor.((wbounds.pMax - wbounds.pMin)/(size+depth-1)) .+ 1) >= N_voxels
    return size + depth - 1
end

calc_voxel_size (generic function with 2 methods)

In [9]:
size = calc_voxel_size(wbounds, N_voxels)

85.0

In [10]:
X, Y, Z = floor.((wbounds.pMax - wbounds.pMin)/size) .+1

3-element Main.RayTracing.Pnt3 with indices SOneTo(3):
 16.0
  4.0
 16.0

In [11]:
# VoxelStruct members
# X, Y, Z, size, wbounds, 
# Dict{Tuple{Int64, Int64, Int64}, Int64} integer coords
# key: tuple integers represent the bottom left coord
# value: light index

In [15]:
n_shadow_rays = 3
voxels = Dict{Tuple{Int64, Int64, Int64}, Int64}()
S = RayTracing.IndependentSampler()
for x in 1:X
    for y in 1:Y
        for z in 1:Z
            light_samples = zeros(Float64, length(lights))

            lcorner = RayTracing.Pnt3(x-1, y-1, z-1) .* size
            center = lcorner .+ size/2

            isect = RayTracing.empty_surface_interation()
            isect.core.p = center

            for idx in eachindex(lights)
                LL = RayTracing.spectrum_from_float(0.0)
                for n in n_shadow_rays
                    u1 = RayTracing.get_2D!(S)
                    u2 = RayTracing.get_2D!(S)
                    # need to write an estimate_direct but simpler
                    # LL += RayTracing.estimate_direct(
                    #     isect,
                    #     u1,
                    #     lights[idx],
                    #     u2,
                    #     scene,
                    #     S,
                    # )
                end
                light_samples[idx] += y(LL) / n_shadow_rays
            end

            u3 = RayTracing.get_1d!(S)
            i, _, _ = RayTracing.sample_discrete(RayTracing.Distribution1D(light_samples), u3)

            voxels[(x,y,z)] = i
        end
    end
end

MethodError: MethodError: objects of type Nothing are not callable